In [ ]:
#@title 1. Settings
PROJECT_DIR     = "" #@param {type:"string"}
DRIVE_ROOT      = "/content/drive/MyDrive" #@param {type:"string"}
PROJECT_FOLDER  = "KYNU truncated isoform" #@param {type:"string"}
BUILDS_ROOT     = "/content/drive/MyDrive/MDs/KYNU_truncated_isoform" #@param {type:"string"}
SETTLED_A       = 2.5 #@param {type:"number"}
#@markdown A cofactor whose displacement stays under this is seated. The value is a
#@markdown judgement about this pocket, not a threshold the paper rests on, and the
#@markdown series is written out so the shape can be read directly.
BURN_FRACTION   = 0.2 #@param {type:"number"}
GMX_VER         = "2026.3" #@param {type:"string"}
GMX_SIMD        = "AVX2_256" #@param ["AVX2_256", "AVX_512", "SSE4.1", "AUTO"]
CACHE_GROMACS   = True #@param {type:"boolean"}
MAXWARN         = 10 #@param {type:"integer"}
WALL_BUDGET_HOURS = 23.0
NS_PER_BUILD = TOTAL_NS = NS_PER_CHUNK = 0

import os, shutil, time
T_START = time.time()
PROJECT = PROJECT_DIR.strip() or os.path.join(DRIVE_ROOT, PROJECT_FOLDER)
MD = os.path.join(PROJECT, "md")
BUILD_ORDER = ["canonical", "mixed"]
print("project", PROJECT)


In [ ]:
#@title 2. Drive, GPU and GROMACS
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, glob, re, subprocess, shutil, datetime, math, collections
import numpy as np

PREFIX = "/usr/local/gromacs"
os.environ["PATH"] = PREFIX + "/bin:" + os.environ["PATH"]
RESULTS = os.path.join(MD, "results")
INPUTS = os.path.join(MD, "inputs")
DATA = os.path.join(PROJECT, "data")
os.makedirs(RESULTS, exist_ok=True)

FIGURES = os.path.join(PROJECT, "figures")
os.makedirs(FIGURES, exist_ok=True)

CG_MARKER = "step3_input.pdb"
CG_REQUIRED = ["step3_input.gro", "step3_input.pdb", "step3_input.psf", "topol.top",
               "index.ndx", "step4.0_minimization.mdp", "step5_production.mdp"]


def find_run_dirs(roots, builds):
    found = []
    for root in roots:
        if not os.path.isdir(root):
            continue
        for dirpath, dirnames, filenames in os.walk(root):
            if CG_MARKER in filenames:
                if dirpath not in [d for _, d, _ in found]:
                    found.append((root, dirpath,
                                  os.path.relpath(dirpath, root).lower()))
                dirnames[:] = []
    out = {}
    for b in builds:
        per_root = [(r, [d for rr, d, rel in found if rr == r and b in rel])
                    for r in roots]
        per_root = [(r, h) for r, h in per_root if h]
        if not per_root:
            raise SystemExit(
                "STOP: no folder holding a %s carries the word %s below any root "
                "searched, %s. Every folder holding a %s: %s."
                % (CG_MARKER, b, roots, CG_MARKER,
                   [d for _, d, _ in found] or "none"))
        root, hits = per_root[0]
        if len(hits) != 1:
            raise SystemExit(
                "STOP: %d folders under %s carry the word %s, %s. One per build is "
                "needed and this is not guessed." % (len(hits), root, b, sorted(hits)))
        out[b] = hits[0]
        for _, other in per_root[1:]:
            for o in other:
                for f in ("step3_input.pdb", "step3_input.gro"):
                    pa, po = os.path.join(out[b], f), os.path.join(o, f)
                    a = os.path.getsize(pa) if os.path.isfile(pa) else None
                    c = os.path.getsize(po) if os.path.isfile(po) else None
                    if a != c:
                        raise SystemExit(
                            "STOP: the %s build is in two places and the copies are "
                            "not the same system. %s has a %s of %s bytes and %s has "
                            "%s. Delete whichever is stale; which one runs is not "
                            "guessed." % (b, out[b], f, a, o, c))
                print("  note: %s is also at %s, same size, and that copy is not used"
                      % (b, o))
    return out


RUN_DIR = find_run_dirs([BUILDS_ROOT, MD], BUILD_ORDER)
print("\nCHARMM-GUI folders:")
for b in BUILD_ORDER:
    miss = [f for f in CG_REQUIRED if not os.path.isfile(os.path.join(RUN_DIR[b], f))]
    extra = "" if not miss else "   MISSING %s" % miss
    print("  %-10s %s%s" % (b, RUN_DIR[b], extra))
    if miss:
        raise SystemExit("STOP: %s is missing %s, so it is not a finished CHARMM-GUI "
                         "GROMACS folder. Unpack the download again and upload the "
                         "whole gromacs/ folder, toppar included." % (RUN_DIR[b], miss))
if not os.path.isdir(os.path.join(RUN_DIR[BUILD_ORDER[0]], "toppar")):
    raise SystemExit("STOP: no toppar/ beside %s. topol.top includes its .itp files by "
                     "relative path, so grompp fails without it."
                     % os.path.basename(RUN_DIR[BUILD_ORDER[0]]))

print("\n--- GPU ---")
HAVE_GPU = False
GPU_NAME = None
try:
    _g = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                         "--format=csv,noheader"], check=False, text=True,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if _g.returncode == 0 and (_g.stdout or "").strip():
        HAVE_GPU = True
        GPU_NAME = _g.stdout.strip().split("\n")[0]
        print(GPU_NAME)
        print("Chunk times scale with this card, so record it with the run: the same "
              "system takes several times longer on a T4 than on an A100.")
    else:
        print("nvidia-smi exited %d, so the runtime claims a GPU it cannot use."
              % _g.returncode)
        print((_g.stdout or "").strip()[:400])
except FileNotFoundError:
    print("no nvidia-smi, so this runtime has no GPU.")
    print("That is fine for the salvage and compare notebooks: they only call "
          "trjconv, trjcat, distance and rms, all of which run on the CPU.")
    print("A production run needs one. Runtime > Change runtime type > T4 GPU.")


def sh(cmd, cwd=None, label=""):
    print("\n$ " + cmd, flush=True)
    r = subprocess.run(cmd, shell=True, cwd=cwd, text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    lines = (r.stdout or "").split("\n")
    if len(lines) > 60:
        print("\n".join(lines[:25]))
        print("   ... %d lines omitted ..." % (len(lines) - 45))
        print("\n".join(lines[-20:]))
    else:
        print(r.stdout)
    if r.returncode != 0:
        raise RuntimeError("%s FAILED (exit %d). The output above is the reason."
                           % (label or cmd, r.returncode))


def gmx_version():
    try:
        return subprocess.run(["gmx", "--version"], capture_output=True, text=True,
                              timeout=90).stdout
    except Exception:
        return ""


CACHE = os.path.join(PROJECT, "gromacs_cuda_%s.tar.gz" % GMX_VER)
CACHE_STALE = False

if "CUDA" in gmx_version():
    print("\nCUDA gmx already present in this session")
elif CACHE_GROMACS and os.path.isfile(CACHE):
    print("\nrestoring the cached GROMACS from", CACHE)
    try:
        sh("tar xzf '%s' -C /" % CACHE, label="restore cache")
    except Exception as e:
        print("the cache could not be read (%s); another session may be writing it. "
              "Building from source instead." % e)
        shutil.rmtree(PREFIX, ignore_errors=True)
    if "CUDA" not in gmx_version():
        print("the cached build does not run here, rebuilding from source")
        shutil.rmtree(PREFIX, ignore_errors=True)
        CACHE_STALE = True

if "CUDA" not in gmx_version():
    nvcc = subprocess.run("which nvcc", shell=True, capture_output=True,
                          text=True).stdout.strip()
    if not nvcc:
        raise SystemExit("STOP: nvcc not found, so there is no CUDA toolkit to build "
                         "against. Set the runtime to GPU and press Run all again.")
    print("\nbuilding GROMACS %s. 20 to 40 minutes, once per Drive folder." % GMX_VER)
    sh("apt-get -qq update && apt-get -qq install -y cmake build-essential "
       "libfftw3-dev", label="apt")
    if not os.path.isdir("/content/gromacs-%s" % GMX_VER):
        sh("wget -q https://ftp.gromacs.org/gromacs/gromacs-%s.tar.gz && "
           "tar xzf gromacs-%s.tar.gz" % (GMX_VER, GMX_VER), cwd="/content",
           label="download")
    os.makedirs("/content/gromacs-%s/build" % GMX_VER, exist_ok=True)
    sh("cmake .. -DGMX_BUILD_OWN_FFTW=OFF -DGMX_GPU=CUDA -DGMX_SIMD=%s "
       "-DCMAKE_INSTALL_PREFIX=%s -DGMX_DOUBLE=OFF -DBUILD_TESTING=OFF"
       % (GMX_SIMD, PREFIX), cwd="/content/gromacs-%s/build" % GMX_VER, label="cmake")
    sh("make -j$(nproc) && make install", cwd="/content/gromacs-%s/build" % GMX_VER,
       label="make")
    if "CUDA" not in gmx_version():
        raise SystemExit("STOP: the build finished but gmx reports no CUDA support.")
    if CACHE_GROMACS and (CACHE_STALE or not os.path.isfile(CACHE)):
        print("\ncaching the build to Drive so later sessions skip it")
        tmp = "%s.%d.tmp" % (CACHE, os.getpid())
        try:
            sh("tar czf '%s' %s" % (tmp, PREFIX.lstrip("/")), cwd="/", label="cache")
            os.replace(tmp, CACHE)
        except Exception as e:
            print("could not cache the build (%s); this costs the next session a "
                  "rebuild and nothing else." % e)
            if os.path.isfile(tmp):
                os.remove(tmp)
    elif CACHE_GROMACS:
        print("\nanother session already cached a build, leaving it alone")

print("\n" + gmx_version().split("\n")[0])


In [ ]:
#@title 3. Helpers
AA = set("ALA ARG ASN ASP CYS GLN GLU GLY HIS HSD HSE HSP ILE LEU LYS MET PHE PRO "
         "SER THR TRP TYR VAL".split())


def gmx(*args, label="", stdin_text=None, quiet=False, check=True):
    cmd = ["gmx"] + [str(a) for a in args]
    if not quiet:
        print("$ " + " ".join(cmd), flush=True)
    r = subprocess.run(cmd, text=True, input=stdin_text,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    out = r.stdout or ""
    lines = out.split("\n")
    if not quiet:
        if len(lines) > 80:
            print("\n".join(lines[:30]))
            print("   ... %d lines omitted ..." % (len(lines) - 60))
            print("\n".join(lines[-30:]))
        else:
            print(out)
    if r.returncode != 0 and check:
        if quiet:
            print(out[-3000:])
        raise RuntimeError("%s failed (exit %d). The output above is the reason."
                           % (label or cmd[1], r.returncode))
    return r.returncode, out


def must_exist(path, what):
    if not os.path.isfile(path):
        raise RuntimeError("%s reported success but %s was not written." % (what, path))


def read_pdb_atoms(path):
    out = []
    for L in open(path, errors="replace"):
        if not L.startswith(("ATOM", "HETATM")):
            continue
        seg = L[72:76].strip() or L[21].strip() or "?"
        out.append((int(L[6:11]), seg, int(L[22:26]), L[17:20].strip(),
                    L[12:16].strip()))
    return out


def protein_segments(atoms):
    rn = {}
    for ser, seg, ri, name, an in atoms:
        rn.setdefault((seg, ri), name)
    return sorted({seg for (seg, ri), name in rn.items() if name in AA})


def progress(**kw):
    p = os.path.join(RESULTS, "progress_%s.json" % BUILD)
    cur = json.load(open(p)) if os.path.isfile(p) else {}
    cur.update(kw)
    cur["updated"] = datetime.datetime.now().isoformat(timespec="seconds")
    cur["elapsed_hours"] = round((time.time() - T_START) / 3600.0, 2)
    json.dump(cur, open(p, "w"), indent=2)
    return cur


def budget_left():
    return WALL_BUDGET_HOURS - (time.time() - T_START) / 3600.0


def xtc_frames(path):
    if not os.path.isfile(path):
        return None
    r = subprocess.run([os.path.join(PREFIX, "bin", "gmx"), "check", "-f", path],
                       text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    out = r.stdout or ""
    m = re.search(r"^\s*Coords\s+(\d+)", out, re.M)
    if m:
        return int(m.group(1))
    m = re.search(r"Last frame\s+(\d+)", out)
    if m:
        return int(m.group(1)) + 1
    if r.returncode != 0:
        return None
    raise SystemExit(
        "STOP: gmx check read %s and exited 0, but its output carried neither a "
        "Coords row nor a Last frame line, so the frame count is unknown. Fix "
        "this parse rather than letting every chunk be re-run as unsound. The "
        "first lines of the output were:\n%s"
        % (os.path.basename(path), "\n".join(out.split("\n")[:12])))


def xtc_last_time(path):
    if not os.path.isfile(path):
        return None
    r = subprocess.run([os.path.join(PREFIX, "bin", "gmx"), "check", "-f", path],
                       text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    t = None
    for L in (r.stdout or "").split("\n"):
        m = re.search(r"frame\s+\d+\s+time\s+([-\d.eE+]+)", L)
        if m:
            t = float(m.group(1))
    return t


def want_frames(mdp, ns):
    dt = float(mdp_get(mdp, "dt"))
    every = int(mdp_get(mdp, "nstxout-compressed"))
    return int(round(ns * 1000.0 / dt / every))


def ndx_args(group="SOLU"):
    if not os.path.isfile("index.ndx"):
        raise SystemExit(
            "STOP: no index.ndx in %s, so the group %s cannot be resolved. It comes "
            "from the CHARMM-GUI download and is staged in with the other inputs, so "
            "if it is missing here the staging is wrong."
            % (os.getcwd(), group))
    return ("-n", "index.ndx")


def stage_in(run, work, n_chunks, per_chunk):
    os.makedirs(work, exist_ok=True)
    heavy = ("prod_all.xtc", "prod_nojump.xtc", "prod_fit.xtc")
    for name in sorted(os.listdir(run)):
        src = os.path.join(run, name)
        if os.path.isdir(src):
            if name == "toppar":
                shutil.copytree(src, os.path.join(work, name), dirs_exist_ok=True)
            continue
        if re.match(r"prod_\d\d\.xtc$", name) or name in heavy:
            continue
        shutil.copy2(src, os.path.join(work, name))
    kept = 0
    for i in range(1, n_chunks + 1):
        nm = "prod_%02d" % i
        src = os.path.join(run, nm + ".xtc")
        if not (os.path.isfile(src) and os.path.isfile(os.path.join(work, nm + ".gro"))):
            print("  %s absent, so it and every chunk after it is re-run" % nm)
            break
        shutil.copy2(src, os.path.join(work, nm + ".xtc"))
        got = xtc_frames(os.path.join(work, nm + ".xtc"))
        if got is None or got < per_chunk - 1:
            os.remove(os.path.join(work, nm + ".xtc"))
            print("  %s holds %s of %d frames, so it and every chunk after it is "
                  "re-run" % (nm, got, per_chunk))
            break
        kept += 1
    for i in range(kept + 1, n_chunks + 1):
        for ext in (".xtc", ".gro", ".cpt", "_prev.cpt", ".tpr", ".log", ".edr"):
            f = os.path.join(work, "prod_%02d%s" % (i, ext))
            if os.path.isfile(f):
                os.remove(f)
    print("  staged into %s, %d of %d chunks reusable" % (work, kept, n_chunks))
    return kept


DRIVE_PENDING = [0]
DRIVE_SENT = {}


def stage_out(work, run, names):
    for nm in names:
        src = os.path.join(work, nm)
        if not os.path.isfile(src):
            continue
        dst = os.path.join(run, nm)
        shutil.copy2(src, dst)
        got, want = os.path.getsize(dst), os.path.getsize(src)
        if got != want:
            raise SystemExit("STOP: %s is %d bytes in the Drive mount against %d on "
                             "disk, so even the cached copy is short."
                             % (nm, got, want))
        DRIVE_PENDING[0] += want
        DRIVE_SENT[dst] = want


def drive_wait(why="", floor=20.0, mbps=12.0):
    mb = DRIVE_PENDING[0] / 1e6
    DRIVE_PENDING[0] = 0
    if mb <= 0:
        return
    secs = max(floor, mb / mbps)
    print("  waiting %.0f s for Drive to take %.0f MB%s"
          % (secs, mb, " after " + why if why else ""), flush=True)
    os.sync()
    time.sleep(secs)


def drive_verify():
    try:
        from google.colab import drive as _drive
        os.sync()
        print("\nflushing the Drive mount, then checking what Drive actually holds")
        _drive.flush_and_unmount()
        _drive.mount("/content/drive")
    except Exception as _e:
        print("\ncould not flush and remount Drive (%s), so the uploads are "
              "unverified. Leave this tab open for a couple of minutes before "
              "closing it." % _e)
        return
    bad, n = [], 0
    for path, want in sorted(DRIVE_SENT.items()):
        n += 1
        got = os.path.getsize(path) if os.path.isfile(path) else None
        if got != want:
            bad.append((os.path.basename(path), got, want))
    if bad:
        for nm, got, want in bad:
            print("   %-22s %s bytes on Drive against %d written" % (nm, got, want))
        raise SystemExit(
            "STOP: %d of %d files are short or missing on Drive after the flush, so "
            "the upload did not finish. The frames are not lost: open this notebook "
            "again and press Run all, which re-sends what is short and skips every "
            "chunk that is whole." % (len(bad), n))
    print("Drive holds all %d uploaded files at full size. Safe to close the runtime."
          % n)


print("helpers ready, MAXWARN =", MAXWARN)


In [ ]:
#@title 4. Per-residue C-alpha fluctuation of each chain
import json, os, shutil

RMSF_DIR = os.path.join(MD, "results", "rmsf")
os.makedirs(RMSF_DIR, exist_ok=True)

_, HELP = gmx("rmsf", "-h", quiet=True, check=False)
for want in (("-[no]fit", "-fit"), ("-[no]res", "-res")):
    if not any(w in HELP for w in want):
        raise SystemExit("STOP: gmx rmsf in this build lists none of %s in its own "
                         "help, so the flag this needs has been renamed or dropped. "
                         "Read gmx rmsf -h before changing the call." % (want,))

summary = {}
for build in BUILD_ORDER:
    run = RUN_DIR[build]
    need = ["prod_01.tpr", "prod_fit.xtc", "iface.ndx", "prod_ref.pdb"]
    miss = [f for f in need if not os.path.isfile(os.path.join(run, f))]
    if miss:
        raise SystemExit("STOP: %s is missing %s. Part 2 stages those to Drive when "
                         "it finishes." % (run, miss))
    rpath = os.path.join(MD, "results", "%s_report.json" % build)
    if not os.path.isfile(rpath):
        raise SystemExit("STOP: no %s, so neither the length of the run nor which "
                         "chain is the canonical one is known here." % rpath)
    rep = json.load(open(rpath))
    frames, burn, ns = int(rep["frames"]), int(rep["burn"]), float(rep["ns"])
    if frames < 2 or burn >= frames:
        raise SystemExit("STOP: %s reports %d frames with %d of them discarded, "
                         "which leaves nothing to measure." % (rpath, frames, burn))
    spacing_ps = ns * 1000.0 / (frames - 1)
    burn_ps = burn * spacing_ps
    roles = rep["roles"]
    chains = [("sub1", roles["canonical"]), ("sub2", roles["partner"])]

    work = os.path.join("/content/work4", build)
    os.makedirs(work, exist_ok=True)
    for f in need:
        if not os.path.isfile(os.path.join(work, f)):
            shutil.copy2(os.path.join(run, f), os.path.join(work, f))
    os.chdir(work)

    ca = {}
    pos = 0
    for L in open(os.path.join(work, "prod_ref.pdb"), errors="replace"):
        if not L.startswith(("ATOM", "HETATM")):
            continue
        pos += 1
        if L[12:16].strip() != "CA":
            continue
        seg = L[72:76].strip() or L[21].strip() or "?"
        ca.setdefault(seg, []).append((int(L[22:26]), L[17:21].strip(), pos))
    built_residue = int(roles["scored_copy"][1])
    for label, seg in chains:
        if seg not in ca:
            raise SystemExit("STOP: prod_ref.pdb of %s carries no C-alpha atom in "
                             "segment %s, and the segments it does carry are %s."
                             % (build, seg, sorted(ca)))
        want = int(roles["lengths"][seg])
        if len(ca[seg]) != want:
            raise SystemExit("STOP: segment %s of %s holds %d C-alpha atoms and the "
                             "run report calls that chain %d residues long, so one "
                             "of the two is reading a different system."
                             % (seg, build, len(ca[seg]), want))
        odd = sorted({(ri, nm) for ri, nm, p in ca[seg] if nm not in AA})
        if any(ri != built_residue for ri, nm in odd):
            raise SystemExit("STOP: segment %s of %s carries a C-alpha in %s, and "
                             "the only residue of a chain that is not a standard "
                             "amino acid should be the one the builder merged with "
                             "the cofactor, at %d."
                             % (seg, build, odd, built_residue))
    with open("rmsf.ndx", "w") as fh:
        for label, seg in chains:
            fh.write("[ %s_ca ]\n" % label)
            ser = [p for ri, nm, p in ca[seg]]
            for i in range(0, len(ser), 15):
                fh.write(" ".join(str(x) for x in ser[i:i + 15]) + "\n")

    summary[build] = {"burn_ps": burn_ps, "spacing_ps": spacing_ps,
                      "frames_total": frames, "frames_scored": frames - burn,
                      "ns": ns, "chains": {}}
    for label, seg in chains:
        xvg = "rmsf_%s.xvg" % label
        if os.path.isfile(xvg):
            os.remove(xvg)
        gmx("rmsf", "-s", "prod_01.tpr", "-f", "prod_fit.xtc", "-n", "rmsf.ndx",
            "-o", xvg, "-res", "-fit", "-b", "%.3f" % burn_ps,
            stdin_text="%s_ca\n" % label,
            label="C-alpha fluctuation of %s in %s" % (label, build), quiet=True)
        must_exist(xvg, "the fluctuation of %s in %s" % (label, build))
        rows = []
        for L in open(xvg):
            if L.startswith(("#", "@")):
                continue
            f = L.split()
            if len(f) >= 2:
                rows.append((int(float(f[0])), float(f[1])))
        if len(rows) != len(ca[seg]):
            raise SystemExit("STOP: %s carries %d residues and segment %s holds %d "
                             "C-alpha atoms, so gmx measured a set other than the "
                             "one it was given."
                             % (xvg, len(rows), seg, len(ca[seg])))
        gmx_ids = [r[0] for r in rows]
        if gmx_ids != sorted(gmx_ids) or len(set(gmx_ids)) != len(gmx_ids):
            raise SystemExit("STOP: the residue column of %s is not strictly "
                             "increasing, so its rows cannot be paired with the "
                             "chain in file order." % xvg)
        tsv = "rmsf_%s.tsv" % label
        vals = []
        with open(tsv, "w") as fh:
            fh.write("residue\tresname\tgmx_residue\trmsf_A\n")
            for (ri, nm, pos), (gi, val) in zip(ca[seg], rows):
                fh.write("%d\t%s\t%d\t%.4f\n" % (ri, nm, gi, val * 10.0))
                vals.append(val * 10.0)
        lo, hi = min(vals), max(vals)
        if not (0.0 < lo and hi < 50.0):
            raise SystemExit("STOP: the fluctuation of %s in %s runs from %.2f to "
                             "%.2f angstroms, which is not a protein chain in a "
                             "finished run." % (label, build, lo, hi))
        summary[build]["chains"][label] = {
            "segment": seg, "residues": len(vals),
            "first_residue": ca[seg][0][0], "last_residue": ca[seg][-1][0],
            "rmsf_A_mean": sum(vals) / len(vals), "rmsf_A_min": lo, "rmsf_A_max": hi}
        dest = os.path.join(RMSF_DIR, build)
        os.makedirs(dest, exist_ok=True)
        for name in (xvg, tsv):
            for d in (run, dest):
                shutil.copy2(name, os.path.join(d, name))

spath = os.path.join(MD, "results", "rmsf_summary.json")
json.dump(summary, open(spath, "w"), indent=2)
os.sync()

print()
for build in BUILD_ORDER:
    s = summary[build]
    print("%s: %d of %d frames scored, from %.0f ps"
          % (build, s["frames_scored"], s["frames_total"], s["burn_ps"]))
    for label in ("sub1", "sub2"):
        c = s["chains"][label]
        print("   %-5s segment %-4s %3d residues %4d to %-4d  mean %.2f A, "
              "%.2f to %.2f" % (label, c["segment"], c["residues"],
                                c["first_residue"], c["last_residue"],
                                c["rmsf_A_mean"], c["rmsf_A_min"], c["rmsf_A_max"]))
print()
print("wrote rmsf_sub1 and rmsf_sub2, xvg and tsv, into each run folder and into")
print(RMSF_DIR)
print("and the summary into %s" % spath)
